# 01 — Data Exploration & Model Training

**PPE Compliance Agent — ITAI 1378 Final Project**

Covers: dataset download, sample inspection, and fine-tuning YOLOv8n on the PPE mask-detection dataset.

Run in Google Colab with a GPU runtime (Runtime > Change runtime type > T4 GPU).

## 1. Install dependencies

In [ ]:
!pip install -q ultralytics roboflow
import ultralytics
ultralytics.checks()

## 2. Mount Google Drive (to persist trained weights)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
SAVE_DIR = "/content/drive/MyDrive/ITAI1378_PPE_Agent"
os.makedirs(SAVE_DIR, exist_ok=True)

## 3. Download dataset from Roboflow

In [ ]:
import roboflow
roboflow.login()
rf = roboflow.Roboflow()
project = rf.workspace("agh-ett2f").project("mask-detection-yolov8")
# confirm the current version number on the Roboflow project page before running
dataset = project.version(16).download("yolov8")
print("Dataset downloaded to:", dataset.location)

## 4. Inspect sample images and class distribution

In [ ]:
import glob
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

sample_images = glob.glob(f"{dataset.location}/train/images/*.jpg")[:6]
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, img_path in zip(axes.flatten(), sample_images):
    ax.imshow(mpimg.imread(img_path))
    ax.axis("off")
plt.tight_layout()
plt.show()

!cat {dataset.location}/data.yaml

## 5. Fine-tune YOLOv8n

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=75,
    imgsz=640,
    batch=16,
    patience=15,
    save_period=10,
    cache=True,
    project="runs/train",
    name="yolov8n_ppe"
)

## 6. Evaluate on the validation split

In [ ]:
metrics = model.val()
print("mAP@0.5:", metrics.box.map50)
print("mAP@0.5:0.95:", metrics.box.map)
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)

## 7. Save trained weights for the agent to use

In [ ]:
import shutil, os

best_weights = "runs/train/yolov8n_ppe/weights/best.pt"

# Save to Drive (persists across sessions)
shutil.copy(best_weights, f"{SAVE_DIR}/best_yolov8n_ppe.pt")

# Save into the repo's expected location so agents/ppe_compliance_agent.py finds it directly
os.makedirs("models/trained", exist_ok=True)
shutil.copy(best_weights, "models/trained/best_yolov8n_ppe.pt")
print("Saved weights to Drive and to models/trained/best_yolov8n_ppe.pt")

## Notes
- If Colab disconnects mid-training, checkpoints save every 10 epochs (`save_period=10`) inside `runs/train/yolov8n_ppe/weights/`. Resume with `model = YOLO('runs/train/yolov8n_ppe/weights/last.pt')` and `.train(resume=True)`.
- `models/trained/best_yolov8n_ppe.pt` is what `agents/ppe_compliance_agent.py` looks for by default. Without it, the agent falls back to base `yolov8n.pt` automatically (see docs/architecture.md).